# NB04 — Airbnb Gold Snowflake Schema

**Technologies:** Microsoft Fabric | PySpark | Delta Lake  
**Data Layer:** Silver → Gold

## 1. Objective

Transform validated Silver datasets into a **business-ready Snowflake Schema** for analysing Melbourne Airbnb listings, indicative prices and advertised availability across LGAs.

The notebook creates three Dimension Tables and one Availability Fact Table, with validation to ensure data consistency and relationship integrity.

## 2. Data Sources & Outputs

| Type | Table |
|---|---|
| Silver Input | `dbo.silver_listings` |
| Silver Input | `dbo.silver_neighbourhoods` |
| Silver Input | `dbo.silver_calendar_90d` |
| Gold Outputs | `dbo.dim_neighbourhood` |
| | `dbo.dim_listing` |
| | `dbo.dim_date` |
| | `dbo.fact_availability` |

## 3. Data Grain & Keys

| Gold Table | Data Grain | Key |
|---|---|---|
| `dim_neighbourhood` | One row per LGA | `neighbourhood_id` |
| `dim_listing` | One row per listing | `listing_id` |
| `dim_date` | One row per date | `date_key` |
| `fact_availability` | One row per listing per date | `listing_id` + `date_key` |

**Design Decision:** Use consistent dimension keys to connect listing, geographic and date information while preserving the Listing-Date grain of the Fact Table.

## 4. Gold Transformation Process

1. **Read Silver Data:** Load validated Listings, Neighbourhoods and 90-day Calendar tables.
2. **Neighbourhood Dimension:** Assign stable surrogate keys (`neighbourhood_id`), preserving existing IDs and adding new LGAs through Delta MERGE.
3. **Listing Dimension:** Join Listings with standardised LGAs and retain listing attributes, price quotes and neighbourhood keys.
4. **Date Dimension:** Generate `date_key` (`YYYYMMDD`), calendar attributes, weekend indicators and configured Victorian public holiday flags.
5. **Availability Fact:** Join Calendar with Listing and Date dimensions to create `fact_availability`.
6. **Gold Storage:** Save the validated analytical tables in Delta format.

## 5. Data Modelling & Business Logic

The model separates descriptive attributes from daily availability records:

- **Neighbourhood:** LGA names and geographic identifiers.
- **Listing:** Room types, property characteristics and indicative prices.
- **Date:** Calendar attributes, weekends and public holidays.
- **Availability Fact:** Listing-Date availability (`is_available`) and minimum/maximum stay requirements.

**Business Logic:** `is_available = 1` represents a listing advertised as available on a specific date, not a confirmed booking or actual occupancy.

Weekend and public holiday indicators support comparisons across different date categories. Holiday dates are configured for the 2026 analysis window.

## 6. Gold Data Quality Validation

The notebook validates:

- Unique and non-null Dimension Keys
- No duplicate Listing-Date Fact Keys
- No unmatched Listing, Neighbourhood or Date Keys
- Complete 90-day coverage for all Listings
- Consistent row counts between Silver Calendar and Gold Fact
- Valid `is_available` values (`0/1`)
- Consistent LGA assignments between Listing and Fact Tables

### Validated Output — September 2026

| Metric | Result |
|---|---:|
| Neighbourhoods | 30 |
| Listings | 25,715 |
| Dates | 90 |
| Availability Fact Rows | 2,314,350 |
| Duplicate Fact Keys | 0 |
| Unmatched Dimension Keys | 0 |
| Incomplete Listing Coverage | 0 |
| Mismatched Listing-LGA Keys | 0 |

**Result:** All Gold data quality validations passed.

## 7. Pipeline Dependency

**NB02 Listings → NB01 Calendar → NB04 Gold**

**NB03 Neighbourhoods → NB04 Gold**

Gold ETL runs after the required Silver datasets are successfully processed.

The resulting Delta tables provide the analytical foundation for the Fabric Semantic Model and downstream dashboard reporting.

In [ ]:
# import Required Libraries
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable


StatementMeta(, cd12cd47-bb9b-4480-a181-04b4c42f356b, 12, Finished, Available, Finished, False)

In [11]:

# Read validated Silver tables
listings = spark.table("dbo.silver_listings")
neighbourhoods = spark.table("dbo.silver_neighbourhoods")
calendar = spark.table("dbo.silver_calendar_90d")

print("Listings:", listings.count())
print("Neighbourhoods:", neighbourhoods.count())
print("Calendar:", calendar.count())


StatementMeta(, cd12cd47-bb9b-4480-a181-04b4c42f356b, 13, Finished, Available, Finished, False)

Listings: 25715
Neighbourhoods: 30
Calendar: 2314350


In [12]:
table_name = "dbo.dim_neighbourhood"

# Step 1: Prepare source data
source_lga = neighbourhoods.select("lga", "lga_key")

source_count = source_lga.count()

assert source_count > 0
assert source_lga.filter(
    F.col("lga").isNull() |
    (F.trim(F.col("lga")) == "") |
    F.col("lga_key").isNull() |
    (F.trim(F.col("lga_key")) == "")
).count() == 0

assert source_lga.select("lga_key").distinct().count() == source_count

# Step 2: Check whether the Gold table already exists
if not spark.catalog.tableExists(table_name):

    # First run: generate initial surrogate keys
    window_spec = Window.orderBy("lga_key")

    dim_neighbourhood = (
        source_lga
        .withColumn(
            "neighbourhood_id",
            F.row_number().over(window_spec)
        )
        .select("neighbourhood_id", "lga", "lga_key")
    )

    (
        dim_neighbourhood.write
        .format("delta")
        .mode("errorifexists")
        .saveAsTable(table_name)
    )

    print("Initial Gold dimension created!")

else:

    # Step 3: Read existing IDs
    existing_dim = spark.table(table_name)

    # Validate existing dimension
    existing_count = existing_dim.count()

    assert existing_dim.select("lga_key").distinct().count() == existing_count
    assert existing_dim.select("neighbourhood_id").distinct().count() == existing_count

    # Keep old mapping for validation
    old_mapping = {
        row["lga_key"]: row["neighbourhood_id"]
        for row in existing_dim.select(
            "lga_key", "neighbourhood_id"
        ).collect()
    }

    # Step 4: Identify NEW LGAs only
    new_lga = source_lga.join(
        existing_dim.select("lga_key"),
        on="lga_key",
        how="left_anti"
    )

    new_count = new_lga.count()

    print("New LGAs found:", new_count)

    if new_count > 0:

        # Step 5: Start new IDs after existing maximum ID
        max_id = existing_dim.agg(
            F.max("neighbourhood_id")
        ).first()[0]

        window_spec = Window.orderBy("lga_key")

        new_rows = (
            new_lga
            .withColumn(
                "neighbourhood_id",
                (
                    F.row_number().over(window_spec)
                    + F.lit(max_id)
                ).cast(existing_dim.schema["neighbourhood_id"].dataType)
            )
            .select("neighbourhood_id", "lga", "lga_key")
        )

        # Step 6: Insert new LGAs without replacing old ones
        delta_table = DeltaTable.forName(
            spark, table_name
        )

        (
            delta_table.alias("target")
            .merge(
                new_rows.alias("source"),
                "target.lga_key = source.lga_key"
            )
            .whenNotMatchedInsert(
                values={
                    "neighbourhood_id": "source.neighbourhood_id",
                    "lga": "source.lga",
                    "lga_key": "source.lga_key"
                }
            )
            .execute()
        )

    # Refresh table after MERGE
    spark.catalog.refreshTable(table_name)

    dim_neighbourhood = spark.table(table_name)

    # Step 7: Verify all existing IDs are unchanged
    current_mapping = {
        row["lga_key"]: row["neighbourhood_id"]
        for row in dim_neighbourhood.select(
            "lga_key", "neighbourhood_id"
        ).collect()
    }

    for lga_key, old_id in old_mapping.items():
        assert current_mapping[lga_key] == old_id, (
            f"ID changed for {lga_key}!"
        )

    print("All existing surrogate keys preserved!")


# Step 8: Final data quality validation
dim_neighbourhood = spark.table(table_name)

total_lga = dim_neighbourhood.count()

unique_ids = dim_neighbourhood.select(
    "neighbourhood_id"
).distinct().count()

unique_lga = dim_neighbourhood.select(
    "lga_key"
).distinct().count()

null_keys = dim_neighbourhood.filter(
    F.col("neighbourhood_id").isNull() |
    F.col("lga_key").isNull()
).count()

assert total_lga > 0
assert unique_ids == total_lga
assert unique_lga == total_lga
assert null_keys == 0

print("Total LGAs:", total_lga)
print("Gold Neighbourhood validation passed!")

display(dim_neighbourhood.orderBy("neighbourhood_id"))
  

StatementMeta(, cd12cd47-bb9b-4480-a181-04b4c42f356b, 14, Finished, Available, Finished, False)

New LGAs found: 0
All existing surrogate keys preserved!
Total LGAs: 30
Gold Neighbourhood validation passed!


SynapseWidget(Synapse.DataFrame, e93e24fb-6bf5-4698-96e9-10214adfa60c)

In [ ]:
# Step 3: Create Gold Listing Dimension

# Link Listings to the Neighbourhood dimension using standardised LGA keys


# Read Silver Listings and Gold Neighbourhood
listings = spark.table("dbo.silver_listings")

neighbourhoods = (
    spark.table("dbo.dim_neighbourhood")
    .select(
        "lga_key",
        "neighbourhood_id",
        F.col("lga").alias("standard_lga")
    )
)

# Standardise LGA names for joining
listings_clean = (
    listings
    .withColumn(
        "lga_key",
        F.lower(
            F.regexp_replace(
                F.trim(F.col("lga")),
                r"\s+",
                " "
            )
        )
    )
)

# Build Listing Dimension
dim_listing = (
    listings_clean
    .join(
        neighbourhoods,
        on="lga_key",
        how="left"
    )
    .select(
        "listing_id",
        "listing_name",
        "host_id",
        "room_type",
        "property_type",
        "accommodates",
        "latitude",
        "longitude",

        # New LGA attributes
        F.col("standard_lga").alias("lga"),
        "neighbourhood_id",

        "listing_price_aud",
        "quote_price_aud",
        "quote_checkin_date",
        "quote_checkout_date",
        "snapshot_date",
        "has_valid_quote"
    )
)

# Validate
source_count = listings.count()
total_listings = dim_listing.count()

unique_listings = (
    dim_listing.select("listing_id")
    .distinct()
    .count()
)

missing_keys = dim_listing.filter(
    F.col("listing_id").isNull() |
    F.col("neighbourhood_id").isNull() |
    F.col("lga").isNull()
).count()

assert source_count > 0
assert total_listings == source_count
assert unique_listings == total_listings
assert missing_keys == 0

print("Total Listings:", total_listings)
print("Missing Keys:", missing_keys)
print("Gold Listing validation passed!")

# Save updated Gold Dimension
(
    dim_listing.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("dbo.dim_listing")
)

print("Gold dim_listing updated!")

display(
    dim_listing.select(
        "listing_id",
        "lga",
        "neighbourhood_id",
        "room_type",
        "quote_price_aud"
    ).limit(10)
)


StatementMeta(, cd12cd47-bb9b-4480-a181-04b4c42f356b, 15, Finished, Available, Finished, False)

Total Listings: 25715
Missing Keys: 0
Gold Listing validation passed!
Gold dim_listing updated!


SynapseWidget(Synapse.DataFrame, e667082a-9f7a-4ae4-91be-01f8dc4ec12c)

In [14]:
# Step 4: Create Gold Date Dimension

calendar = spark.table("dbo.silver_calendar_90d")

# Create one record per unique calendar date
dim_date = (
    calendar
    .select("date")
    .distinct()

    # Create stable date key: YYYYMMDD
    .withColumn(
        "date_key",
        F.date_format("date", "yyyyMMdd").cast("int")
    )

    # Extract date attributes
    .withColumn("year", F.year("date"))
    .withColumn("quarter", F.quarter("date"))
    .withColumn("month", F.month("date"))
    .withColumn("month_name", F.date_format("date", "MMMM"))
    .withColumn("day_of_month", F.dayofmonth("date"))
    .withColumn("day_name", F.date_format("date", "EEEE"))

    # Weekend: Sunday = 1, Saturday = 7
    .withColumn(
        "is_weekend",
        F.when(
            F.dayofweek("date").isin(1, 7), 1
        ).otherwise(0)
    )

    # Victorian public holidays within the 2026 snapshot
    .withColumn(
        "public_holiday_name",
        F.when(
            F.col("date") == F.to_date(F.lit("2026-09-25")),
            "Friday before AFL Grand Final"
        )
        .when(
            F.col("date") == F.to_date(F.lit("2026-11-03")),
            "Melbourne Cup Day"
        )
    )

    # Holiday flag: 1 = public holiday, 0 = other day
    .withColumn(
        "is_public_holiday",
        F.when(
            F.col("public_holiday_name").isNotNull(), 1
        ).otherwise(0)
    )

    # Optional: Weekend OR public holiday
    .withColumn(
        "is_weekend_or_holiday",
        F.when(
            (F.col("is_weekend") == 1) |
            (F.col("is_public_holiday") == 1),
            1
        ).otherwise(0)
    )

    .select(
        "date_key",
        "date",
        "year",
        "quarter",
        "month",
        "month_name",
        "day_of_month",
        "day_name",
        "is_weekend",
        "is_public_holiday",
        "public_holiday_name",
        "is_weekend_or_holiday"
    )
)

# Validate before saving
date_count = dim_date.count()

unique_date_keys = (
    dim_date.select("date_key").distinct().count()
)

null_dates = dim_date.filter(
    F.col("date").isNull() |
    F.col("date_key").isNull()
).count()

print("Total Dates:", date_count)
print("Unique Date Keys:", unique_date_keys)
print("Null Dates:", null_dates)

# 90 days is the required analysis window
assert date_count == 90
assert unique_date_keys == date_count
assert null_dates == 0

# Save Gold Date Dimension
(
    dim_date.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("dbo.dim_date")
)

print("Gold dim_date created successfully!")

# Display holiday records
display(
    dim_date.filter(
        F.col("is_public_holiday") == 1
    ).orderBy("date")
)


StatementMeta(, cd12cd47-bb9b-4480-a181-04b4c42f356b, 16, Finished, Available, Finished, False)

Total Dates: 90
Unique Date Keys: 90
Null Dates: 0
Gold dim_date created successfully!


SynapseWidget(Synapse.DataFrame, dddaa077-1c5a-4116-987f-630bb0f78c14)

In [15]:
# Step 5: Create Gold Availability Fact Table

calendar = spark.table("dbo.silver_calendar_90d")
dim_listing = spark.table("dbo.dim_listing")
dim_date = spark.table("dbo.dim_date")

# Get the neighbourhood key from Gold dim_listing
listing_keys = dim_listing.select(
    "listing_id",
    "neighbourhood_id"
)

# Get the date key from Gold dim_date
date_keys = dim_date.select("date", "date_key")

# Build Gold Fact Table
fact_availability = (
    calendar
    .join(
        listing_keys,
        on="listing_id",
        how="left"
    )
    .join(
        date_keys,
        on="date",
        how="left"
    )
    .select(
        "listing_id",
        "date_key",
        "neighbourhood_id",
        "is_available",
        "minimum_nights",
        "maximum_nights"
    )
)

# Save Gold Availability Fact Table
(
    fact_availability.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("dbo.fact_availability")
)

print("Gold fact_availability updated successfully!")

StatementMeta(, cd12cd47-bb9b-4480-a181-04b4c42f356b, 17, Finished, Available, Finished, False)

Gold fact_availability updated successfully!



## **4. Relationship Validation — Listings vs Neighbourhoods**

### Objective
Verify that every Airbnb listing can be matched to a
valid Local Government Area (LGA).

### Method
A standardised `lga_key` is created in the Listings
dataset using the same transformation rules.

A LEFT ANTI JOIN is used to identify listings that
do not have a matching LGA in the Neighbourhood table.

### Validation Results
- Total Listings: 25,715
- Unmatched Listings: 0
- LGA Match Rate: 100%

### Business Impact
All listings can be assigned to a valid LGA.

This supports accurate aggregation of listing supply,
room-type distribution, indicative prices and
advertised availability by geographic area.

### Data Modelling
The Neighbourhood table contains one record per LGA,
while the Listings table can contain multiple listings
within the same LGA.

This forms a one-to-many relationship.



In [ ]:
# Step 6: Gold Layer Validation



# Read saved Gold tables
dim_l = spark.table("dbo.dim_listing")
dim_n = spark.table("dbo.dim_neighbourhood")
dim_d = spark.table("dbo.dim_date")
fact = spark.table("dbo.fact_availability")

# 1. Validate Dimension Primary Keys
def check_unique_key(df, key):
    total = df.count()
    unique = df.select(key).distinct().count()
    nulls = df.filter(F.col(key).isNull()).count()

    print(f"{key}: {total} rows, {unique} unique, {nulls} nulls")

    if total != unique or nulls > 0:
        raise ValueError(f"Invalid primary key: {key}")

check_unique_key(dim_l, "listing_id")
check_unique_key(dim_n, "neighbourhood_id")
check_unique_key(dim_d, "date_key")

# 2. Validate Fact Composite Key
duplicate_facts = (
    fact
    .groupBy("listing_id", "date_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Duplicate Fact Keys:", duplicate_facts)

if duplicate_facts > 0:
    raise ValueError("Duplicate listing-date records detected.")

# 3. Check Foreign Key relationships
relationships = [
    ("listing_id", dim_l),
    ("neighbourhood_id", dim_n),
    ("date_key", dim_d)
]

for key, dimension in relationships:
    missing = (
        fact.select(key).distinct()
        .join(
            dimension.select(key),
            on=key,
            how="left_anti"
        )
        .count()
    )

    print(f"Unmatched {key} keys:", missing)

    if missing > 0:
        raise ValueError(f"Foreign key validation failed: {key}")

# 4. Check Row Count
silver_rows = spark.table("dbo.silver_calendar_90d").count()
gold_rows = fact.count()

print("Silver Calendar Rows:", silver_rows)
print("Gold Fact Rows:", gold_rows)

if silver_rows != gold_rows:
    raise ValueError("Silver and Gold row counts do not match.")

# 4A. Validate 90-day listing coverage

silver_listings = spark.table("dbo.silver_listings")
silver_calendar = spark.table("dbo.silver_calendar_90d")

listing_ids = silver_listings.select("listing_id").distinct()
calendar_ids = silver_calendar.select("listing_id").distinct()

# Listings missing from Calendar
missing_listings = (
    listing_ids
    .join(calendar_ids, "listing_id", "left_anti")
    .count()
)

# Calendar Listings not present in Listings snapshot
unexpected_listings = (
    calendar_ids
    .join(listing_ids, "listing_id", "left_anti")
    .count()
)

# Check every Listing has exactly 90 unique dates
coverage = (
    silver_calendar
    .groupBy("listing_id")
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("date").alias("date_count")
    )
)

incomplete_listings = coverage.filter(
    (F.col("row_count") != 90) |
    (F.col("date_count") != 90)
).count()

print("Listings missing from Calendar:", missing_listings)
print("Unexpected Calendar Listings:", unexpected_listings)
print("Listings without 90 complete days:", incomplete_listings)

assert missing_listings == 0, "Missing Calendar Listings!"
assert unexpected_listings == 0, "Unexpected Calendar Listings!"
assert incomplete_listings == 0, "Incomplete 90-day coverage!"

print("90-day Calendar coverage validation passed!")


# 5. Validate Availability Flag
invalid_flags = fact.filter(
    F.col("is_available").isNull() |
    (~F.col("is_available").isin(0, 1))
).count()

print("Invalid Availability Flags:", invalid_flags)

if invalid_flags > 0:
    raise ValueError("Invalid availability flag detected.")


# 6. Validate Listings vs Neighbourhoods

silver_listings = spark.table("dbo.silver_listings")
silver_neighbourhoods = spark.table("dbo.silver_neighbourhoods")

# Standardise LGA keys in Listings
listing_lga = silver_listings.withColumn(
    "lga_key",
    F.lower(
        F.regexp_replace(
            F.trim(F.col("lga")),
            r"\s+",
            " "
        )
    )
)

# Find Listings without a matching LGA
unmatched_lga = (
    listing_lga
    .join(
        silver_neighbourhoods.select("lga_key"),
        on="lga_key",
        how="left_anti"
    )
)

unmatched_count = unmatched_lga.count()

print("Unmatched Listings to LGAs:", unmatched_count)

if unmatched_count > 0:
    raise ValueError(
        "Some Listings cannot be matched to an LGA!"
    )

print("Listing-LGA relationship validation passed!")

print("All Gold Layer validations passed!")


StatementMeta(, cd12cd47-bb9b-4480-a181-04b4c42f356b, 18, Finished, Available, Finished, False)

listing_id: 25715 rows, 25715 unique, 0 nulls
neighbourhood_id: 30 rows, 30 unique, 0 nulls
date_key: 90 rows, 90 unique, 0 nulls
Duplicate Fact Keys: 0
Unmatched listing_id keys: 0
Unmatched neighbourhood_id keys: 0
Unmatched date_key keys: 0
Silver Calendar Rows: 2314350
Gold Fact Rows: 2314350
Listings missing from Calendar: 0
Unexpected Calendar Listings: 0
Listings without 90 complete days: 0
90-day Calendar coverage validation passed!
Invalid Availability Flags: 0
Unmatched Listings to LGAs: 0
Listing-LGA relationship validation passed!
All Gold Layer validations passed!


In [17]:
# Check neighbourhood consistency between
# Listing Dimension and Availability Fact

mismatched_lga = (
    fact.alias("f")
    .join(
        dim_l.select(
            "listing_id", "neighbourhood_id"
        ).alias("l"),
        on="listing_id",
        how="inner"
    )
    .filter(
        F.col("f.neighbourhood_id") !=
        F.col("l.neighbourhood_id")
    )
    .count()
)

print("Mismatched Listing-LGA Keys:", mismatched_lga)

assert mismatched_lga == 0

StatementMeta(, cd12cd47-bb9b-4480-a181-04b4c42f356b, 19, Finished, Available, Finished, False)

Mismatched Listing-LGA Keys: 0
